## 📚 Book Recommendation System
#### A content-based book recommendation system built using TF-IDF Vectorizer 
#### and Cosine Similarity on a dataset of 6000+ books.

 #### 📌 Table of Contents
1. Importing Libraries
2. Loading Dataset
3. Data Cleaning
4. Feature Engineering (Tags)
5. Vectorization (TF-IDF)
6. Cosine Similarity
7. Recommendation Function
8. Saving Model

## 1. Importing Libraries

In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

## 2. Loading Dataset

In [2]:
books = pd.read_csv(r"C:\Users\aaadi\Desktop\data.csv")

In [3]:
books.head()

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0
3,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0
4,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0


## 3. Data Cleaning

In [4]:
books.shape

(6810, 12)

In [5]:
books = books[["title", "description", "authors", "categories", "thumbnail"]] 

In [6]:
books.head()

,title,description,authors,categories,thumbnail
0,Gilead,A NOVEL THAT READERS and critics have been eag...,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...
1,Spider's Web,A new 'Christie for Christmas' -- a full-lengt...,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...
2,The One Tree,Volume Two of Stephen Donaldson's acclaimed se...,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...
3,Rage of angels,"A memorable, mesmerizing heroine Jennifer -- b...",Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...
4,The Four Loves,Lewis' work on the nature of love divides love...,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...


In [7]:
books.isnull().sum()

title            0
description    262
authors         72
categories      99
thumbnail      329
dtype: int64

In [8]:
books = books.dropna(subset = ["description","authors","categories","thumbnail"])

In [9]:
books.isnull().sum()

title          0
description    0
authors        0
categories     0
thumbnail      0
dtype: int64

In [10]:
books = books.reset_index(drop=True)

In [11]:
books["authors"] = books["authors"].apply(lambda x:x.replace(" ", ""))
books["categories"] = books["categories"].apply(lambda x:x.replace(" ", ""))
books["authors"] = books["authors"].apply(lambda x:x.replace(";", ","))

In [12]:
books.dtypes

title          object
description    object
authors        object
categories     object
thumbnail      object
dtype: object

In [13]:
books["description"] = books["description"].apply(lambda x: x.split())

In [14]:
books["description"]

0       [A, NOVEL, THAT, READERS, and, critics, have, ...
1       [A, new, 'Christie, for, Christmas', --, a, fu...
2       [Volume, Two, of, Stephen, Donaldson's, acclai...
3       [A, memorable,, mesmerizing, heroine, Jennifer...
4       [Lewis', work, on, the, nature, of, love, divi...
                              ...                        
6242    [This, book, tells, the, tale, of, a, man, who...
6243    [Wisdom, to, Create, a, Life, of, Passion,, Pu...
6244    [This, collection, of, the, timeless, teaching...
6245    [Since, the, three, volume, edition, ofHegel's...
6246    [This, is, a, jubilant, and, rewarding, collec...
Name: description, Length: 6247, dtype: object

In [15]:
type(books["categories"][0])

str

In [16]:
books.head()

,title,description,authors,categories,thumbnail
0,Gilead,"[A, NOVEL, THAT, READERS, and, critics, have, ...",MarilynneRobinson,Fiction,http://books.google.com/books/content?id=KQZCP...
1,Spider's Web,"[A, new, 'Christie, for, Christmas', --, a, fu...","CharlesOsborne,AgathaChristie",Detectiveandmysterystories,http://books.google.com/books/content?id=gA5GP...
2,The One Tree,"[Volume, Two, of, Stephen, Donaldson's, acclai...",StephenR.Donaldson,Americanfiction,http://books.google.com/books/content?id=OmQaw...
3,Rage of angels,"[A, memorable,, mesmerizing, heroine, Jennifer...",SidneySheldon,Fiction,http://books.google.com/books/content?id=FKo2T...
4,The Four Loves,"[Lewis', work, on, the, nature, of, love, divi...",CliveStaplesLewis,Christianlife,http://books.google.com/books/content?id=XhQ5X...


In [17]:
books["authors"] = books["authors"].apply(lambda x: [x])
books["categories"] = books["categories"].apply(lambda x: [x])

In [18]:
books.head()

,title,description,authors,categories,thumbnail
0,Gilead,"[A, NOVEL, THAT, READERS, and, critics, have, ...",[MarilynneRobinson],[Fiction],http://books.google.com/books/content?id=KQZCP...
1,Spider's Web,"[A, new, 'Christie, for, Christmas', --, a, fu...","[CharlesOsborne,AgathaChristie]",[Detectiveandmysterystories],http://books.google.com/books/content?id=gA5GP...
2,The One Tree,"[Volume, Two, of, Stephen, Donaldson's, acclai...",[StephenR.Donaldson],[Americanfiction],http://books.google.com/books/content?id=OmQaw...
3,Rage of angels,"[A, memorable,, mesmerizing, heroine, Jennifer...",[SidneySheldon],[Fiction],http://books.google.com/books/content?id=FKo2T...
4,The Four Loves,"[Lewis', work, on, the, nature, of, love, divi...",[CliveStaplesLewis],[Christianlife],http://books.google.com/books/content?id=XhQ5X...


## 4. Feature Engineering (Tags)


In [19]:
books["tags"] = books["description"] + books["authors"] + books["categories"]


In [20]:
books.head()


,title,description,authors,categories,thumbnail,tags
0,Gilead,"[A, NOVEL, THAT, READERS, and, critics, have, ...",[MarilynneRobinson],[Fiction],http://books.google.com/books/content?id=KQZCP...,"[A, NOVEL, THAT, READERS, and, critics, have, ..."
1,Spider's Web,"[A, new, 'Christie, for, Christmas', --, a, fu...","[CharlesOsborne,AgathaChristie]",[Detectiveandmysterystories],http://books.google.com/books/content?id=gA5GP...,"[A, new, 'Christie, for, Christmas', --, a, fu..."
2,The One Tree,"[Volume, Two, of, Stephen, Donaldson's, acclai...",[StephenR.Donaldson],[Americanfiction],http://books.google.com/books/content?id=OmQaw...,"[Volume, Two, of, Stephen, Donaldson's, acclai..."
3,Rage of angels,"[A, memorable,, mesmerizing, heroine, Jennifer...",[SidneySheldon],[Fiction],http://books.google.com/books/content?id=FKo2T...,"[A, memorable,, mesmerizing, heroine, Jennifer..."
4,The Four Loves,"[Lewis', work, on, the, nature, of, love, divi...",[CliveStaplesLewis],[Christianlife],http://books.google.com/books/content?id=XhQ5X...,"[Lewis', work, on, the, nature, of, love, divi..."


In [21]:
books = books[["title","tags","thumbnail"]]

In [22]:
books.head()

,title,tags,thumbnail
0,Gilead,"[A, NOVEL, THAT, READERS, and, critics, have, ...",http://books.google.com/books/content?id=KQZCP...
1,Spider's Web,"[A, new, 'Christie, for, Christmas', --, a, fu...",http://books.google.com/books/content?id=gA5GP...
2,The One Tree,"[Volume, Two, of, Stephen, Donaldson's, acclai...",http://books.google.com/books/content?id=OmQaw...
3,Rage of angels,"[A, memorable,, mesmerizing, heroine, Jennifer...",http://books.google.com/books/content?id=FKo2T...
4,The Four Loves,"[Lewis', work, on, the, nature, of, love, divi...",http://books.google.com/books/content?id=XhQ5X...


In [23]:
books["tags"] = books["tags"].apply(lambda x: [i.lower() for i in x])

In [24]:
books.head()

,title,tags,thumbnail
0,Gilead,"[a, novel, that, readers, and, critics, have, ...",http://books.google.com/books/content?id=KQZCP...
1,Spider's Web,"[a, new, 'christie, for, christmas', --, a, fu...",http://books.google.com/books/content?id=gA5GP...
2,The One Tree,"[volume, two, of, stephen, donaldson's, acclai...",http://books.google.com/books/content?id=OmQaw...
3,Rage of angels,"[a, memorable,, mesmerizing, heroine, jennifer...",http://books.google.com/books/content?id=FKo2T...
4,The Four Loves,"[lewis', work, on, the, nature, of, love, divi...",http://books.google.com/books/content?id=XhQ5X...


In [25]:
books["tags"] = books["tags"].apply(lambda x: [i.replace("'", "") for i in x])


In [26]:
books.head()

,title,tags,thumbnail
0,Gilead,"[a, novel, that, readers, and, critics, have, ...",http://books.google.com/books/content?id=KQZCP...
1,Spider's Web,"[a, new, christie, for, christmas, --, a, full...",http://books.google.com/books/content?id=gA5GP...
2,The One Tree,"[volume, two, of, stephen, donaldsons, acclaim...",http://books.google.com/books/content?id=OmQaw...
3,Rage of angels,"[a, memorable,, mesmerizing, heroine, jennifer...",http://books.google.com/books/content?id=FKo2T...
4,The Four Loves,"[lewis, work, on, the, nature, of, love, divid...",http://books.google.com/books/content?id=XhQ5X...


In [27]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aaadi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [28]:
stop_words = set(stopwords.words("english"))

In [29]:
def remove_stopwords(text):
    filtered = [word for word in text if word not in stop_words]
    return filtered
    

In [30]:
books["tags"] = books["tags"].apply(remove_stopwords)

In [31]:
books.head()

,title,tags,thumbnail
0,Gilead,"[novel, readers, critics, eagerly, anticipatin...",http://books.google.com/books/content?id=KQZCP...
1,Spider's Web,"[new, christie, christmas, --, full-length, no...",http://books.google.com/books/content?id=gA5GP...
2,The One Tree,"[volume, two, stephen, donaldsons, acclaimed, ...",http://books.google.com/books/content?id=OmQaw...
3,Rage of angels,"[memorable,, mesmerizing, heroine, jennifer, -...",http://books.google.com/books/content?id=FKo2T...
4,The Four Loves,"[lewis, work, nature, love, divides, love, fou...",http://books.google.com/books/content?id=XhQ5X...


In [32]:
books["tags"] = books["tags"].apply(lambda x: [i.replace(".", "") for i in x])

In [33]:
books["tags"] = books["tags"].apply(lambda x: [i.replace(" ", "") for i in x])
books["tags"] = books["tags"].apply(lambda x: [i.replace(",", "") for i in x])
books["tags"] = books["tags"].apply(lambda x: [i.replace("-", "") for i in x])

In [34]:
books["tags"] = books["tags"].apply(lambda x: [i.replace("(", "") for i in x])
books["tags"] = books["tags"].apply(lambda x: [i.replace(")", "") for i in x])

In [35]:
books.tail()

,title,tags,thumbnail
6242,Journey to the East,"[book, tells, tale, man, goes, wonderful, amaz...",http://books.google.com/books/content?id=rq6JP...
6243,The Monk Who Sold His Ferrari: A Fable About F...,"[wisdom, create, life, passion, purpose, peace...",http://books.google.com/books/content?id=c_7mf...
6244,I Am that,"[collection, timeless, teachings, one, greates...",http://books.google.com/books/content?id=Fv_JP...
6245,The Berlin Phenomenology,"[since, three, volume, edition, ofhegels, phil...",http://books.google.com/books/content?id=Vy7Sk...
6246,'I'm Telling You Stories',"[jubilant, rewarding, collection, winterson, s...",http://books.google.com/books/content?id=2lVyR...


## 5. Vectorization (TF-IDF)

In [36]:
books["tags"] = books["tags"].apply(lambda x: " ".join(x))

In [37]:
tfidf = TfidfVectorizer(max_features = 5000)
vectors = tfidf.fit_transform(books['tags'])

In [38]:
vectors.shape

(6247, 5000)

In [39]:
#Vectorization using CountVectorizer-
#from sklearn.feature_extraction.text import CountVectorizer
#c1 = CountVectorizer(max_features = 5000)
#vectors = c1.fit_transform(books["tags"]).toarray()

## 6. Cosine Similarity

In [40]:
similarity = cosine_similarity(vectors)

## 7. Recommendation Function

In [41]:
def recommend(name):
    book_index = books[books["title"] == name].index[0]
    des = similarity[book_index]
    des = list(enumerate(des))
    des = sorted(des, key = lambda x: x[1], reverse = True)
    top5 = des[1:6]
    for i in top5:
        print(books.iloc[i[0]].title)
    
    

## 8. Sample Outputs

In [42]:
recommend("I Am that")

Wolf's Hour
The Buddha in Your Mirror
The Beloved
Existentialists and Mystics
Kaddish and Other Poems: 1958-1960


In [43]:
recommend("The Power of Now")

Journey To Ixtlan
The Complete Dream Dictionary
I Ching
Blessings from the Other Side
History of Philosophy


In [44]:
recommend("Think and Grow Rich")

Think and Grow Rich: The 21st-Century Edition
Hard Drive
Execution
Liberals and Communitarians
Naked Economics: Undressing the Dismal Science


In [46]:
pickle.dump(similarity, open('similarity.pkl', 'wb'))
pickle.dump(books, open('books.pkl', 'wb'))